In [ ]:
# %pip install sentence-transformers

In [ ]:
from sentence_transformers import SentenceTransformer

In [ ]:
model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

In [ ]:
questions = [
    "What is overfitting?",
    "How can you identify whether a machine learning model is overfitting?",
    "What is the difference between precision and recall?",
    "How do precision and recall differ?",
    "Explain what a SQL JOIN does.",
    "Why would you use a SQL JOIN?"
]

In [ ]:
embeddings = model.encode(
    questions,
    normalize_embeddings=True
)

print(embeddings.shape)

In [ ]:
import numpy as np

similarity_matrix = np.matmul(
    embeddings,
    embeddings.T
)

print(similarity_matrix)

In [ ]:
def cosine_similarity(
    embedding_a,
    embedding_b
):
    return float(
        np.dot(
            embedding_a,
            embedding_b
        )
    )

In [ ]:
score = cosine_similarity(
    embeddings[0],
    embeddings[1]
)

print(score)

In [ ]:
SIMILARITY_THRESHOLD = 0.80

def is_semantically_similar(
    new_question: str,
    previous_questions: list[str],
    threshold: float = SIMILARITY_THRESHOLD
) -> bool:

    if not previous_questions:
        return False

    new_embedding = model.encode(
        new_question,
        normalize_embeddings=True
    )

    previous_embeddings = model.encode(
        previous_questions,
        normalize_embeddings=True
    )

    similarities = np.matmul(
        previous_embeddings,
        new_embedding
    )

    return bool(
        np.max(similarities) >= threshold
    )

In [ ]:
previous = [
    "What is overfitting?"
]

print(
    is_semantically_similar(
        "How can you identify whether a model is overfitting?",
        previous
    )
)

In [ ]:
print(
    is_semantically_similar(
        "Explain SQL JOINs.",
        previous
    )
)

In [ ]:
similarity_tests = [
    {
        "q1": "What is overfitting?",
        "q2": "How can you identify overfitting in a machine learning model?",
        "expected": True
    },
    {
        "q1": "What is precision?",
        "q2": "What is recall?",
        "expected": False
    },
    {
        "q1": "Explain SQL JOINs.",
        "q2": "Why would you use a JOIN in SQL?",
        "expected": True
    },
    {
        "q1": "What is cross validation?",
        "q2": "Why do we use cross validation when training models?",
        "expected": True
    },
    {
        "q1": "What is gradient descent?",
        "q2": "Explain the purpose of train-test splitting.",
        "expected": False
    }
]

In [ ]:
for test in similarity_tests:

    e1 = model.encode(
        test["q1"],
        normalize_embeddings=True
    )

    e2 = model.encode(
        test["q2"],
        normalize_embeddings=True
    )

    score = cosine_similarity(e1, e2)

    print(
        f"{score:.3f} | "
        f"Expected: {test['expected']} | "
        f"{test['q1']}  ↔  {test['q2']}"
    )

In [ ]:
# %pip install faiss-cpu
import faiss

In [ ]:
dimension = embeddings.shape[1]

index = faiss.IndexFlatIP(
    dimension
)

index.add(
    embeddings.astype("float32")
)

In [ ]:
new_question = (
    "How would you detect if your ML model is overfitting?"
)

new_embedding = model.encode(
    [new_question],
    normalize_embeddings=True
).astype("float32")

new_embedding = model.encode(
    [new_question],
    normalize_embeddings=True
).astype("float32")

scores, indices = index.search(
    new_embedding,
    k=3
)

for score, idx in zip(
    scores[0],
    indices[0]
):

    print(
        f"{score:.3f} -> {questions[idx]}"
    )

In [ ]:
class QuestionSimilarity:

    def __init__(
        self,
        model_name="all-MiniLM-L6-v2"
    ):

        self.model = SentenceTransformer(
            model_name
        )

    def encode(
        self,
        questions: list[str]
    ):

        return self.model.encode(
            questions,
            normalize_embeddings=True
        )

    def similarity(
        self,
        question_a: str,
        question_b: str
    ):

        embeddings = self.encode(
            [question_a, question_b]
        )

        return float(
            np.dot(
                embeddings[0],
                embeddings[1]
            )
        )
        


In [ ]:
similarity_service = QuestionSimilarity()

score = similarity_service.similarity(
    "What is overfitting?",
    "How do you identify overfitting?"
)

print(score)

In [ ]:
score = similarity_service.similarity(
    "What is overfitting?",
    "How does a SQL JOIN work?"
)

print(score)

In [ ]:
question_bank = [
    "What is overfitting?",
    "How can you detect overfitting in a machine learning model?",
    "What is underfitting?",
    "How does regularization help prevent overfitting?",
    
    "What is precision?",
    "What is recall?",
    "When would you prefer precision over recall?",
    "What is the F1 score?",
    
    "What is cross validation?",
    "Why do we use cross validation?",
    
    "What is gradient descent?",
    "How does gradient descent optimize a model?",
    
    "What is a SQL JOIN?",
    "Explain the difference between INNER JOIN and LEFT JOIN.",
    
    "What is normalization in machine learning?",
    "Why is feature scaling important?",
]

In [ ]:
question_embeddings = model.encode(
    question_bank,
    normalize_embeddings=True
)

print(question_embeddings.shape)

In [ ]:
dimension = question_embeddings.shape[1]

index = faiss.IndexFlatIP(
    dimension
)

index.add(
    question_embeddings.astype("float32")
)

print("Questions in index:", index.ntotal)


In [ ]:
def search_similar_questions(
    query: str,
    k: int = 5
):
    
    query_embedding = model.encode(
        [query],
        normalize_embeddings=True
    ).astype("float32")

    scores, indices = index.search(
        query_embedding,
        k
    )

    results = []

    for score, idx in zip(
        scores[0],
        indices[0]
    ):
        results.append({
            "question": question_bank[idx],
            "similarity": float(score)
        })

    return results

In [ ]:
results = search_similar_questions(
    "How would you know if your model is overfitting?",
    k=5
)

for result in results:
    print(
        f"{result['similarity']:.3f}",
        "→",
        result["question"]
    )

In [ ]:
DUPLICATE_THRESHOLD = 0.80

def is_duplicate_question(
    new_question: str,
    threshold: float = DUPLICATE_THRESHOLD
) -> bool:

    results = search_similar_questions(
        new_question,
        k=1
    )

    if not results:
        return False

    return (
        results[0]["similarity"]
        >= threshold
    )

In [ ]:
print(
    is_duplicate_question(
        "How can you identify overfitting?"
    )
)
print(
    is_duplicate_question(
        "What is gradient descent?"
    )
)
print(
    is_duplicate_question(
        "How would you design a recommendation system?"
    )
)

In [ ]:
asked_questions = [
    "What is overfitting?",
    "Explain the difference between precision and recall.",
    "Why do we use cross validation?"
]



In [ ]:
def check_question_history(
    new_question: str,
    asked_questions: list[str],
    threshold: float = 0.80
):

    if not asked_questions:
        return {
            "is_duplicate": False,
            "similarity": 0.0,
            "matched_question": None
        }

    new_embedding = model.encode(
        [new_question],
        normalize_embeddings=True
    ).astype("float32")

    history_embeddings = model.encode(
        asked_questions,
        normalize_embeddings=True
    ).astype("float32")

    scores = np.matmul(
        history_embeddings,
        new_embedding[0]
    )

    best_index = int(
        np.argmax(scores)
    )

    best_score = float(
        scores[best_index]
    )

    return {
        "is_duplicate": best_score >= threshold,
        "similarity": best_score,
        "matched_question": asked_questions[best_index]
    }

In [ ]:
result = check_question_history(
    "How can you identify overfitting in a model?",
    asked_questions
)

print(result)

In [ ]:
import json

with open(
    "../data/question_bank/questions.json",
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        question_bank,
        f,
        indent=2,
        ensure_ascii=False
    )